In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import pandas as pd
from sklearn.model_selection import GridSearchCV
import numpy as np
from sklearn.metrics import f1_score 

import joblib 

import os

In [2]:
X_train_lr = pd.read_parquet('X_train_lr.parquet')
y_train_lr = pd.read_parquet('y_train_lr.parquet')
X_train_lr_smote = pd.read_parquet('X_train_lr_smote.parquet')
y_train_lr_smote = pd.read_parquet('y_train_lr_smote.parquet')
X_val_lr = pd.read_parquet('X_val_lr.parquet')
y_val_lr = pd.read_parquet('y_val_lr.parquet')
X_test_lr = pd.read_parquet('X_test_lr.parquet')
y_test_lr = pd.read_parquet('y_test_lr.parquet')

### Methodology 
First, a model will be trained on the non-smote data. Some GridSearchCV will be also used.

In [3]:
# Create the model
lr_model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight='balanced'      
)

param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["lbfgs"]
}

grid = GridSearchCV(
    estimator= LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1
)

grid.fit(X_train_lr, y_train_lr)

best_model = grid.best_estimator_

print(grid.best_params_)



c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


{'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}


In [4]:
os.makedirs('models', exist_ok=True)

In [5]:
joblib.dump(grid.best_estimator_, "models/logistic_regression_best.pkl")

['models/logistic_regression_best.pkl']

In [6]:
#Find the best threshold for the model using the validation set
y_val_prob = best_model.predict_proba(X_val_lr)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.01)

best_threshold = 0.5
best_f1 = 0

for threshold in thresholds:

    y_val_pred = (y_val_prob >= threshold).astype(int)

    f1 = f1_score(y_val_lr, y_val_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f'Best threshold: {best_threshold}')
print(f'Best F1-score: {best_f1}')

Best threshold: 0.15999999999999998
Best F1-score: 0.29190151290418276


In [7]:
#Evalkuate the model on the test set using the best threshold

y_test_prob = best_model.predict_proba(X_test_lr)[:, 1]

y_test_pred = (y_test_prob >= best_threshold).astype(int)

print("Accuracy :", accuracy_score(y_test_lr, y_test_pred))
print("Precision:", precision_score(y_test_lr, y_test_pred))
print("Recall   :", recall_score(y_test_lr, y_test_pred))
print("F1-score :", f1_score(y_test_lr, y_test_pred))
print("ROC AUC  :", roc_auc_score(y_test_lr, y_test_prob))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_lr, y_test_pred))

print("\nClassification Report")
print(classification_report(y_test_lr, y_test_pred))

Accuracy : 0.8578118140578508
Precision: 0.24431818181818182
Recall   : 0.3637462235649547
F1-score : 0.2923039572711823
ROC AUC  : 0.7471138323423325

Confusion Matrix
[[50952  5586]
 [ 3159  1806]]

Classification Report
              precision    recall  f1-score   support

           0       0.94      0.90      0.92     56538
           1       0.24      0.36      0.29      4965

    accuracy                           0.86     61503
   macro avg       0.59      0.63      0.61     61503
weighted avg       0.89      0.86      0.87     61503



#### Train on SMOTE data 
Same steps will be used, except for the class_weight = 'balanced' parameter of LR, which will be removed.

In [8]:
# Create the model
lr_model_sm = LogisticRegression(
    random_state=42,
    max_iter=1000   
)

param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["lbfgs"]
}

grid_sm = GridSearchCV(
    estimator= LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1
)

grid_sm.fit(X_train_lr_smote, y_train_lr_smote)

best_model_sm = grid_sm.best_estimator_

print("\nBest parameters")
print(grid_sm.best_params_)

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)



Best parameters
{'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}


In [9]:
#Find the best threshold for the model using the validation set
y_val_prob_sm = best_model_sm.predict_proba(X_val_lr)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.01)

best_threshold_sm = 0.5
best_f1_sm = 0

for threshold in thresholds:

    y_val_pred = (y_val_prob_sm >= threshold).astype(int)

    f1_sm = f1_score(y_val_lr, y_val_pred)

    if f1_sm > best_f1_sm:
        best_f1_sm = f1_sm
        best_threshold_sm = threshold

print(f'Best threshold: {best_threshold_sm}')
print(f'Best F1-score: {best_f1_sm}')

Best threshold: 0.6799999999999997
Best F1-score: 0.28507635973921586


In [10]:
#Evaluate the model on the test set using the best threshold

y_test_prob_sm = best_model_sm.predict_proba(X_test_lr)[:, 1]

y_test_pred_sm = (y_test_prob_sm >= best_threshold_sm).astype(int)

print("Accuracy :", accuracy_score(y_test_lr, y_test_pred_sm))
print("Precision:", precision_score(y_test_lr, y_test_pred_sm))
print("Recall   :", recall_score(y_test_lr, y_test_pred_sm))
print("F1-score :", f1_score(y_test_lr, y_test_pred_sm))
print("ROC AUC  :", roc_auc_score(y_test_lr, y_test_prob_sm))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_lr, y_test_pred_sm))

print("\nClassification Report")
print(classification_report(y_test_lr, y_test_pred_sm))

Accuracy : 0.8400240638668033
Precision: 0.22654847396768402
Recall   : 0.4066465256797583
F1-score : 0.29098508323124594
ROC AUC  : 0.7411176049745367

Confusion Matrix
[[49645  6893]
 [ 2946  2019]]

Classification Report
              precision    recall  f1-score   support

           0       0.94      0.88      0.91     56538
           1       0.23      0.41      0.29      4965

    accuracy                           0.84     61503
   macro avg       0.59      0.64      0.60     61503
weighted avg       0.89      0.84      0.86     61503



In [11]:
joblib.dump(grid_sm.best_estimator_, "models/logistic_regression_smote_best.pkl")

['models/logistic_regression_smote_best.pkl']